In [1]:
import pandas as pd
import numpy as np

# preprocessing
from sklearn.model_selection import train_test_split,RandomizedSearchCV
from sklearn.preprocessing import StandardScaler,OneHotEncoder
from imblearn.under_sampling import RandomUnderSampler
from scipy.stats import loguniform

# model machine learning
from sklearn.compose import ColumnTransformer
# from sklearn.pipeline import Pipeline
from imblearn.pipeline import Pipeline
from sklearn.feature_selection import SequentialFeatureSelector
from sklearn.ensemble import AdaBoostClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score,log_loss
import warnings
warnings.filterwarnings('ignore', category=UserWarning)

# 1.Binary Classification AdaBoost

In [2]:
df = pd.read_csv('../0.dataset/dataset_Heart_attack_clean.csv',nrows=5000)
df_x = df.drop(columns='heart_attack')
df_y = df['heart_attack']

In [3]:
feature_numerik = ['age','body_mass_index','systolic_blood_pressure','forced_expiratory_volume_1','time_to_event_or_censoring']
feature_categori = ['gender']

preprocessor = ColumnTransformer(
    transformers=[
        ('numerik_scaler',StandardScaler(),feature_numerik),
        ('categori_encoding',OneHotEncoder(),feature_categori)
        ],
   remainder='passthrough' # Kolom kategori yang sudah ter-encode dari sananya akan aman lewat lewat sini
)

In [4]:
selector_model = AdaBoostClassifier()
model_ComplementNB = Pipeline([
    ('preprocessing', preprocessor),
    ('undersampling',RandomUnderSampler(random_state=42)),
    ('feature_selection', SequentialFeatureSelector(estimator=selector_model, direction='forward')),
    ('model_classifier', AdaBoostClassifier())
])

In [ ]:
params = {
    'feature_selection__n_features_to_select':[5],
    'model_classifier__n_estimators': [50, 100, 200, 300], # Jumlah pohon/weak learner
    'model_classifier__learning_rate': [0.01, 0.05, 0.1, 0.5, 1.0],
}
X_train,X_test,y_train,y_test = train_test_split(df_x,df_y,test_size=0.2,random_state=42,stratify=df_y)

random_search = RandomizedSearchCV(estimator=model_ComplementNB,param_distributions=params,n_iter=2,cv=5,scoring='accuracy',random_state=42,n_jobs=-1,verbose=3)
random_search.fit(X_train,y_train)